# DeepStreetSearch — text and image search
Search a saved panorama index with **text** or an **image**. Image search also shows the geographic head's direct location prediction.

Set the checkpoint, index, and grid below, then run all cells. For images, enter a local file path and choose whether it is a full 360° panorama or an ordinary photo/crop.

In [ ]:
CHECKPOINT = 'outputs/checkpoints/deepstreetsearch_E3.pt'
INDEX = 'outputs/indexes/deepstreetsearch_E3_index.pt'
GRID = 'outputs/grids/world.aggrid'
GALLERY_SIZE = 20000     # None = full index (may require substantial RAM)
GALLERY_SEED = 42
AGGREGATION = 'max'      # 'max': best matching crops; 'mean': average panorama
TOP_K = 10

In [ ]:
from pathlib import Path
import sys
import torch

root = Path.cwd().resolve()
if not (root / 'modules' / 'inference.py').exists():
    root = root.parent
if not (root / 'modules' / 'inference.py').exists():
    raise FileNotFoundError('Open Jupyter from the project root (or its immediate child).')
sys.path.insert(0, str(root)) if str(root) not in sys.path else None

from config import SIGLIP_MODEL
from modules.inference import GeoSearchInference
from text_location_eval.inspect_results import original_panorama

def local_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else root / path

for name, value in [('CHECKPOINT', CHECKPOINT), ('INDEX', INDEX), ('GRID', GRID)]:
    if not local_path(value).is_file():
        raise FileNotFoundError(f'{name}: {local_path(value)}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
search = GeoSearchInference(
    checkpoint=local_path(CHECKPOINT), index=local_path(INDEX),
    grid=local_path(GRID), model_name=SIGLIP_MODEL,
    device=device, gallery_size=GALLERY_SIZE,
    gallery_seed=GALLERY_SEED, aggregation=AGGREGATION,
)
cache_dir = root / 'outputs/search_cache'
print(f'Ready: {search.gallery_count:,} indexed panoramas | {device.upper()}')

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from PIL import Image

mode = widgets.ToggleButtons(options=['Text', 'Image'], description='Search by:')
text_box = widgets.Textarea(value='A mountain road surrounded by forest',
    description='Text:', layout=widgets.Layout(width='800px', height='75px'))
image_box = widgets.Text(value='path/to/photo.jpg', description='Image path:',
    layout=widgets.Layout(width='800px'))
is_panorama = widgets.Checkbox(value=False, description='Full 360° panorama (6 crops)')
rank_box = widgets.BoundedIntText(value=1, min=1, max=TOP_K, description='Rank:')
button = widgets.Button(description='Search', button_style='primary', icon='search')
output = widgets.Output()
state = {'matches': [], 'geo': [], 'image': None}

def show_match():
    with output:
        clear_output(wait=True)
        matches = state['matches']
        if not matches:
            print('No matches.')
            return
        result = matches[rank_box.value - 1]
        print(f'Rank #{result["rank"]}/{len(matches)} | Similarity: {result["similarity"]:.4f}')
        lat, lon = result['latitude'], result['longitude']
        display(Markdown(f'**Visual match:** [{lat:.5f}, {lon:.5f}](https://www.openstreetmap.org/?mlat={lat}&mlon={lon}#map=13/{lat}/{lon})'))
        if state['image'] is not None:
            with Image.open(state['image']) as query_image:
                query_preview = query_image.copy()
            query_preview.thumbnail((600, 350))
            print('Query image:')
            display(query_preview)
        if state['geo']:
            print('Direct geographic head predictions (independent of the index):')
            for place in state['geo']:
                print(f'  #{place["rank"]}: {place["latitude"]:.4f}, {place["longitude"]:.4f} (p={place["probability"]:.3f})')
        try:
            path = original_panorama(cache_dir, result)
            with Image.open(path) as image:
                preview = image.copy()
            preview.thumbnail((1400, 850))
            display(preview)
        except (FileNotFoundError, ValueError, OSError, KeyError) as error:
            print(f'Panorama image unavailable (original dataset shard required): {error}')

def on_search(_):
    with output:
        clear_output(wait=True)
        print('Searching...')
    try:
        if mode.value == 'Text':
            matches = search.retrieve(text_box.value.strip(), top_k=TOP_K)
            geo = []
            query_image = None
        else:
            path = local_path(image_box.value)
            if not path.is_file():
                raise FileNotFoundError(f'Image not found: {path}')
            matches = search.retrieve_image(path, top_k=TOP_K, panorama=is_panorama.value)
            geo = search.predict_image_location(path, top_k=5, panorama=is_panorama.value)
            query_image = path
        state.update(matches=matches, geo=geo, image=query_image)
        show_match()
    except Exception:
        with output:
            clear_output(wait=True)
            import traceback
            traceback.print_exc()

def on_rank(change):
    if change['name'] == 'value' and state['matches']:
        show_match()

button.on_click(on_search)
rank_box.observe(on_rank, names='value')
display(widgets.VBox([mode, text_box, image_box, is_panorama,
                      widgets.HBox([rank_box, button]), output]))